In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from glob import glob
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

EDA was open-ended, and it was up to me to decide how to look at different ways to slice and dice my data. 

This EDA should also help to inform me of how pneumonia looks in the wild. E.g. what other types of diseases it's commonly found with, how often it is found, what ages it affects, etc. 

Note that this NIH dataset was not specifically acquired for pneumonia. So, while this is a representation of 'pneumonia in the wild,' the prevalence of pneumonia may be different if we were to take only chest x-rays that were acquired in an ER setting with suspicion of pneumonia. 

Perform the following EDA:
* The patient demographic data such as gender, age, patient position,etc. (as it is available)
* The x-ray views taken (i.e. view position)
* The number of cases including: 
    * number of pneumonia cases,
    * number of non-pneumonia cases
* The distribution of other diseases that are comorbid with pneumonia
* Number of disease per patient 
* Pixel-level assessments of the imaging data for healthy & disease states of interest (e.g. histograms of intensity values) and compare distributions across diseases.

Note: use full NIH data to perform the first a few EDA items and use `sample_labels.csv` for the pixel-level assassements. 

I also had to, **describe my findings and how will I set up the model training based on the findings.**

In [ ]:
## Below is some helper code to read data for us.
## Load NIH data
all_xray_df = pd.read_csv('/data/Data_Entry_2017.csv')
all_xray_df.sample(3)

In [ ]:
## Load 'sample_labels.csv' data for pixel level assessments
sample_df = pd.read_csv('sample_labels.csv')
sample_df.sample(3)

## EDA

### a) The patient demographic data

In [ ]:
# Clinical variable EDA — Patient Gender and View Position

# Show types and frequencies for gender and view position

for col in ['Patient Gender', 'View Position']:
    print('\n----', col, '----')
    if col in all_xray_df.columns:
        vc = all_xray_df[col].fillna('Missing').value_counts()
        print(vc)
        # Bar plot
        plt.figure(figsize=(6,3))
        plt.bar(vc.index.astype(str), vc.values)
        plt.title(f'{col} distribution')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{col} not found in dataframe')

In [ ]:
# Clinical variable EDA — Patient Age
# Convert to numeric, show summary stats and a histogram
if 'Patient Age' in all_xray_df.columns:
    ages = pd.to_numeric(all_xray_df['Patient Age'], errors='coerce')
    print('Age summary statistics:')
    print(ages.describe())
    plt.figure(figsize=(6,3))
    plt.hist(ages.dropna(), bins=30)
    plt.xlabel('Patient Age')
    plt.ylabel('Count')
    plt.title('Patient Age distribution')
    plt.show()
else:
    print('Patient Age column not found')

Since there is a patient with a register of 414 year old, and this is not possible (ages < 0 or > 120), I will remove this row.

In [ ]:
# Here I remove rows in the 'Patient Age' column that have missing or invalid values from the dataframe to avoid confusion in later analysis
if 'Patient Age' in all_xray_df.columns:
    all_xray_df = all_xray_df.dropna(subset=['Patient Age'])
    print('Rows with missing or invalid Patient Age removed from dataframe.')
    # Remove outliers in Patient Age (e.g., ages < 0 or > 120)
    all_xray_df = all_xray_df[(all_xray_df['Patient Age'] >= 0) & (all_xray_df['Patient Age'] <= 120)]
    print('Outliers in Patient Age removed from dataframe.')
else:
    print('Patient Age column not found, no rows removed.')

In [ ]:
if 'Patient Age' in all_xray_df.columns:
    ages = pd.to_numeric(all_xray_df['Patient Age'], errors='coerce')
    print('Age summary statistics:')
    print(ages.describe())
    plt.figure(figsize=(6,3))
    plt.hist(ages.dropna(), bins=30)
    plt.xlabel('Patient Age')
    plt.ylabel('Count')
    plt.title('Patient Age distribution')
    plt.show()
else:
    print('Patient Age column not found')

Then, I repeat the exploration for gender and view position once I've removed outliers:

In [ ]:
# Show types and frequencies for gender and view position

for col in ['Patient Gender', 'View Position']:
    print('\n----', col, '----')
    if col in all_xray_df.columns:
        vc = all_xray_df[col].fillna('Missing').value_counts()
        print(vc)
        # Bar plot
        plt.figure(figsize=(6,3))
        plt.bar(vc.index.astype(str), vc.values)
        plt.title(f'Repetition of {col} distribution')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{col} not found in dataframe')

Here I explored the studies per patient in order to know how many studies each patient had.

In [ ]:
# Clinical variable EDA — Studies per patient
if 'Patient ID' in all_xray_df.columns:
    studies_per_patient = all_xray_df['Patient ID'].value_counts()
    print('Top 10 patients by number of studies:')
    print(studies_per_patient.head(10))
    plt.figure(figsize=(6,3))
    plt.hist(studies_per_patient.values, bins=range(1, studies_per_patient.max()+2))
    plt.xlabel('Studies per patient')
    plt.ylabel('Number of patients')
    plt.title('Distribution of studies per patient')
    plt.show()
else:
    print('Patient ID column not found')

Since there are several patients participating in more than 100 studies, I would have a look to what patients participated in more than 100 studies, in 75 to 99 studies, 50 to 74 studies, 25 to 49 studies, and 10 to 24 studies.

In [ ]:
# Clinical variable EDA — # Clinical variable EDA — 
# Show types and frequencies for finding labels
# of patients participating in more than 100 studies, 
# in 75 to 99 studies, 50 to 74 studies, 25 to 49 studies, 
# and 10 to 24 studies.
if 'Patient ID' in all_xray_df.columns:
    studies_per_patient = all_xray_df['Patient ID'].value_counts()
    bins = [0, 10, 25, 50, 75, 100, float('inf')]
    labels = ['<10', '10-24', '25-49', '50-74', '75-99', '100+']
    studies_per_patient_binned = pd.cut(studies_per_patient, bins=bins, labels=labels)
    print('Distribution of patients by number of studies:')
    print(studies_per_patient_binned.value_counts().sort_index())
    plt.figure(figsize=(10,5))
    studies_per_patient_binned.value_counts().sort_index().plot(kind='bar')
    plt.title('Distribution of patients by number of studies')
    plt.xlabel('Number of studies')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

Since this participation in diverse studies could be associated to comorbidities, I need to explore it too.

In [ ]:
# Clinical variable EDA — Finding Labels
# Show types and frequencies for finding labels
# of patients participating in more than 100 studies, 
# in 75 to 99 studies, 50 to 74 studies, 25 to 49 studies, 
# and 10 to 24 studies.

if 'Finding Labels' in all_xray_df.columns:
    finding_labels = all_xray_df['Finding Labels'].fillna('No Finding')
    finding_counts = finding_labels.value_counts()
    print('Finding Labels distribution:')
    print(finding_counts)
    plt.figure(figsize=(10,5))
    plt.bar(finding_counts.index.astype(str), finding_counts.values)
    plt.title('Finding Labels distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=90, ha='right')
    plt.tight_layout()
    plt.show()

Since the graph presented several patients with a unique mixture of labels, I analysed what individual labels are present in the total data and the frequencies:

In [ ]:
# Clinical variable EDA — Finding Labels
# Split the finding labels into individual labels and show their frequencies
if 'Finding Labels' in all_xray_df.columns:
    all_labels = all_xray_df['Finding Labels'].dropna().str.split('|').explode()
    label_counts = all_labels.value_counts()
    print('Individual Finding Labels distribution:')
    print(label_counts)
    plt.figure(figsize=(10,5))
    plt.bar(label_counts.index.astype(str), label_counts.values)
    plt.title('Individual Finding Labels distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=90, ha='right')
    plt.tight_layout()
    plt.show()

I would explore those cases with several annotations; that is which labels are > 2 frequent. Since the graph was not very informative due to there are an important amount of patients with a unique mix of labels, I also explored the top-10 most common finding labels in the case of those that are not unique in patients (we have more than 2 patients with the same label), and with more than 1 label.

In [ ]:
# Clinical variable EDA — Finding Labels
# Show types and frequencies > 2 for finding labels (only labels with '|' before splitting into individual labels)
if 'Finding Labels' in all_xray_df.columns:
    finding_labels = all_xray_df['Finding Labels'].fillna('No Finding')
    finding_counts = finding_labels[finding_labels.str.contains('\|')].value_counts()
    finding_counts_gt1 = finding_counts[finding_counts > 2]
    print('Finding Labels distribution (counts > 2):')
    print(finding_counts_gt1)
    plt.figure(figsize=(10,5))
    plt.bar(finding_counts_gt1.index.astype(str), finding_counts_gt1.values)
    plt.title('Finding Labels distribution (counts > 2)')
    plt.ylabel('Count')
    plt.xticks(rotation=90, ha='right')
    plt.tight_layout()
    plt.show()

# Return top-10 most common finding labels
if 'Finding Labels' in all_xray_df.columns:
    finding_labels = all_xray_df['Finding Labels'].fillna('No Finding')
    finding_counts = finding_labels[finding_labels.str.contains('\|')].value_counts()
    finding_counts_gt1 = finding_counts[finding_counts > 2]
    top_10_labels = finding_counts_gt1.head(10)
    print('Top 10 most common finding labels with more than one label:')
    print(top_10_labels)

### b) The x-ray views taken (i.e. view position)

In [ ]:
# Analysis by labels: frequencies, pneumonia prevalence, and comorbidities
from collections import Counter
# To ensure there are no NaNs and split pipe-separated labels into lists
df = all_xray_df.copy()
df['Finding Labels'] = df['Finding Labels'].fillna('No Finding')
df['label_list'] = df['Finding Labels'].apply(lambda x: [lbl.strip() for lbl in str(x).split('|') if lbl.strip()!=''])
# I count label frequencies across all studies
label_counts = Counter([lbl for sub in df['label_list'] for lbl in sub])
print('Top 20 labels (by study count):')
for lbl, cnt in label_counts.most_common(20):
    print(f'{lbl}: {cnt}')

# Pneumonia prevalence (study-level) - use case-insensitive match
# Updated to be case-insensitive so labels like 'pneumonia' are detected
df['has_pneumonia'] = df['Finding Labels'].str.contains('Pneumonia', case=False, na=False)
total = len(df)
pnum = int(df['has_pneumonia'].sum())
print()
print(f'Total studies: {total}')
print(f'Pneumonia studies: {pnum} ({pnum/total:.2%})')

# Number of labels per study (exclude 'No Finding' as a disease)
df['n_labels'] = df['label_list'].apply(lambda lst: 0 if (len(lst) == 1 and lst[0] == 'No Finding') else len(lst))
plt.figure(figsize=(6,4))
plt.hist(df['n_labels'], bins=range(0, int(df['n_labels'].max())+2), align='left')
plt.xlabel('Number of labels per study')
plt.ylabel('Count')
plt.title('Labels per study')
plt.show()

# Per-patient aggregation: unique labels across all studies of each patient
patient_labels = df.groupby('Patient ID')['label_list'].apply(lambda lists: set([l for sub in lists for l in sub if l != 'No Finding']))
patient_nlabels = patient_labels.apply(len)
print()
print(f'Average unique labels per patient: {patient_nlabels.mean():.2f}')
plt.figure(figsize=(6,4))
plt.hist(patient_nlabels, bins=range(0, int(patient_nlabels.max())+2), align='left')
plt.xlabel('Unique labels per patient')
plt.ylabel('Count')
plt.title('Labels per patient (unique)')
plt.show()



In [ ]:
# Deeper EDA on X-ray View Position
# This cell expands the earlier summary: counts, pneumonia prevalence by view,
# and top comorbid labels (for pneumonia cases) within the most common views.
col = 'View Position'
if col in df.columns:
    view_ser = df[col].fillna('Missing')
    view_counts = view_ser.value_counts()
    print('View position counts:')
    print(view_counts)
    plt.figure(figsize=(8,4))
    plt.bar(view_counts.index.astype(str), view_counts.values)
    plt.xticks(rotation=45, ha='right')
    plt.title('X-ray View Position Counts')
    plt.tight_layout()
    plt.show()

    # Pneumonia prevalence per view position
    agg = df.assign(_view=view_ser).groupby('_view')['has_pneumonia'].agg(total_pneumonia='sum', total_studies='count')
    agg['prevalence'] = agg['total_pneumonia'] / agg['total_studies']
    agg = agg.sort_values('prevalence', ascending=False)
    print('Pneumonia prevalence by view position:')
    print(agg)
    plt.figure(figsize=(8,4))
    plt.bar(agg.index.astype(str), agg['prevalence'])
    plt.title('Pneumonia prevalence by View Position')
    plt.ylabel('Pneumonia prevalence (fraction)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    # Top comorbid labels in pneumonia cases for the top view positions
    top_views = view_counts.head(5).index.tolist()
    for v in top_views:
        sub = df[(view_ser==v) & (df['has_pneumonia'])]
        lbls = Counter([lbl for sublist in sub['label_list'] for lbl in sublist if lbl not in ('Pneumonia','No Finding')])
        plt.figure(figsize=(8,4))
        labels, counts = zip(*lbls.most_common(8))
        plt.bar(labels, counts)
        plt.title(f'Comorbid labels with Pneumonia ({v})')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print(f'{col} not found in dataframe')
    plt.show()
    

In case it would be relevant, I also check other variables, such as age or gender:

In [ ]:
# Clinical variable EDA — Gender and View Position
# Here I show types and frequencies as column plots for gender and view position together
if 'Patient Gender' in all_xray_df.columns and 'View Position' in all_xray_df.columns:
    gender_view_counts = all_xray_df.groupby(['Patient Gender', 'View Position']).size().unstack(fill_value=0)
    print('Gender and View Position distribution:')
    print(gender_view_counts)
    gender_view_counts.plot(kind='bar', stacked=False, figsize=(8,4))
    plt.title('Gender and View Position distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Clinical variable EDA — Age distribution and position
# Here I show types and frequencies as column plotsfor age distribution (as range of ages from 0 to 100, in groups of 10) and view position
if 'Patient Age' in all_xray_df.columns and 'View Position' in all_xray_df.columns:
    all_xray_df['Age Group'] = pd.cut(all_xray_df['Patient Age'], bins=range(0, 110, 10), right=False)
    age_view_counts = all_xray_df.groupby(['Age Group', 'View Position']).size().unstack(fill_value=0)
    print('Age Group and View Position distribution:')
    print(age_view_counts)
    age_view_counts.plot(kind='bar', stacked=False, figsize=(10,5))
    plt.title('Age Group and View Position distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### c) The number of cases

Including comparison of number of pneumonia cases, and number of non-pneumonia cases.

In [ ]:
# Including comparison of number of pneumonia cases, and number of non-pneumonia cases.
pnum_counts = df['has_pneumonia'].value_counts()
plt.figure(figsize=(6,4))
plt.bar(['No Pneumonia', 'Pneumonia'], pnum_counts.values)
plt.ylabel('Count')
plt.title('Pneumonia vs No Pneumonia cases')
plt.show()



I also explored the cases depending on other variables.

In [ ]:
# Clinical variable EDA — Finding Labels and cases
# Here I have types and frequencies for finding labels according to cases (pneumonia vs non-pneumonia)
if 'Finding Labels' in df.columns:
    pneumonia_labels = df[df['has_pneumonia']]['Finding Labels'].fillna('No Finding')
    non_pneumonia_labels = df[~df['has_pneumonia']]['Finding Labels'].fillna('No Finding')
    
    pneumonia_counts = pneumonia_labels.value_counts()
    non_pneumonia_counts = non_pneumonia_labels.value_counts()
    
    print('Finding Labels distribution for Pneumonia cases:')
    print(pneumonia_counts)
    print('\nFinding Labels distribution for Non-Pneumonia cases:')
    print(non_pneumonia_counts)
    
    # Plotting
    plt.figure(figsize=(12,5))
    
    plt.subplot(1, 2, 1)
    plt.bar(pneumonia_counts.index.astype(str), pneumonia_counts.values)
    plt.title('Pneumonia Cases - Finding Labels')
    plt.ylabel('Count')
    plt.xticks(rotation=90, ha='right')
    
    plt.subplot(1, 2, 2)
    plt.bar(non_pneumonia_counts.index.astype(str), non_pneumonia_counts.values)
    plt.title('Non-Pneumonia Cases - Finding Labels')
    plt.ylabel('Count')
    plt.xticks(rotation=90, ha='right')
    
    plt.tight_layout()
    plt.show()

# Clinical variable EDA — Gender and cases
# To show types and frequencies for gender according to cases (pneumonia vs non-pneumonia)
if 'Patient Gender' in df.columns:
    pneumonia_gender = df[df['has_pneumonia']]['Patient Gender'].fillna('Missing')
    non_pneumonia_gender = df[~df['has_pneumonia']]['Patient Gender'].fillna('Missing')
    pneumonia_gender_counts = pneumonia_gender.value_counts()
    non_pneumonia_gender_counts = non_pneumonia_gender.value_counts()
    print('Patient Gender distribution for Pneumonia cases:')
    print(pneumonia_gender_counts)
    print('\nPatient Gender distribution for Non-Pneumonia cases:')
    print(non_pneumonia_gender_counts)

    # Plotting
    plt.figure(figsize=(10,5))
    
    plt.subplot(1, 2, 1)
    plt.bar(pneumonia_gender_counts.index.astype(str), pneumonia_gender_counts.values)
    plt.title('Pneumonia Cases - Patient Gender')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    
    plt.subplot(1, 2, 2)
    plt.bar(non_pneumonia_gender_counts.index.astype(str), non_pneumonia_gender_counts.values)
    plt.title('Non-Pneumonia Cases - Patient Gender')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

### d) The distribution of other diseases that are comorbid with pneumonia

In [ ]:
# Comorbidities that co-occur with Pneumonia
comorbid_counter = Counter()
for labels, has_p in zip(df['label_list'], df['has_pneumonia']):
    if has_p:
        for lbl in labels:
            if lbl != 'Pneumonia' and lbl != 'No Finding':
                comorbid_counter[lbl] += 1
print()
print('Top comorbid labels with Pneumonia:')
for lbl, cnt in comorbid_counter.most_common(15):
    print(f'{lbl}: {cnt}')


# Bar plot of top comorbidities with Pneumonia (matplotlib so it works without seaborn)
top = comorbid_counter.most_common(10)
if top:
    labels, counts = zip(*top)
    plt.figure(figsize=(8,4))
    plt.bar(labels, counts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Top comorbid labels with Pneumonia')
    plt.tight_layout()
    plt.show()

### e) Number of disease per patient 


In [ ]:
# Number of comorbidities per patient (Patient ID, regardless of whether they have Pneumonia or not)
counter_per_patient = df.groupby('Patient ID')['label_list'].apply(lambda lists: set([l for sub in lists for l in sub if l != 'No Finding']))
n_comorbid_per_patient = counter_per_patient.apply(len)
print()
print(f'Average number of comorbidities per patient: {n_comorbid_per_patient.mean():.2f}')
plt.figure(figsize=(6,4))
plt.hist(n_comorbid_per_patient, bins=range(0, int(n_comorbid_per_patient.max())+2), align='left')
plt.xlabel('Number of comorbidities per patient')
plt.ylabel('Count')
plt.title('Comorbidities per patient')
plt.show()

### f) Pixel-level assessments 

Pixel-level assessments of the imaging data for healthy & disease states of interest (e.g. histograms of intensity values) and compare distributions across diseases

In [ ]:
# Normalize / unify a few column name variants that appear across different CSVs
import re
def unify_known_columns(df):
    # map a few common variants (case-insensitive) to canonical names used later in the notebook
    mapping = {}
    for c in df.columns:
        key = re.sub(r'[^0-9a-z]', '', str(c).lower())
        if key in ('originalimagepixelspacingx', 'original_image_pixel_spacing_x', 'originalimagepixelspacing_x'):
            mapping[c] = 'OriginalImagePixelSpacing_x'
        elif key in ('originalimagepixelspacingy', 'original_image_pixel_spacing_y', 'originalimagepixelspacing_y'):
            mapping[c] = 'OriginalImagePixelSpacing_y'
        elif key == 'originalimagewidth':
            mapping[c] = 'OriginalImageWidth'
        elif key == 'originalimageheight':
            mapping[c] = 'OriginalImageHeight'
    if mapping:
        df.rename(columns=mapping, inplace=True)
    return df

def fix_split_bracket_columns(df):
    # Handle headers split into two columns like 'OriginalImagePixelSpacing[x' and 'y]'
    cols = list(df.columns)
    rename_map = {}
    i = 0
    while i < len(cols) - 1:
        a = cols[i]
        b = cols[i+1]
        # detect pattern where a contains '[' and the next column contains ']' or looks like a suffix
        if '[' in a and (']' in b or b.strip().endswith(']')):
            base = a.split('[')[0].strip()
            sa = a.split('[')[1].replace(']', '').strip()
            sb = b.replace(']', '').replace('[', '').strip()
            if base:
                new_a = base + ('_' if not base.endswith('_') else '') + sa
                new_b = base + ('_' if not base.endswith('_') else '') + sb
                rename_map[a] = new_a
                rename_map[b] = new_b
                i += 2
                continue
        i += 1
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
    return df

# Safe drop helper to avoid KeyError when columns may be missing
def safe_drop_cols(df, cols):
    df.drop(columns=[c for c in cols if c in df.columns], inplace=True, errors='ignore')

# Apply normalization to dataframes we load in this notebook
all_xray_df = fix_split_bracket_columns(all_xray_df)
all_xray_df = unify_known_columns(all_xray_df)
# If sample_df is loaded later, call fix_split_bracket_columns(sample_df) and unify_known_columns(sample_df) after loading it
print('Unified column names for all_xray_df (examples):', list(all_xray_df.columns)[:10])

In [ ]:
# Imaging data for healthy and diseased patients assessment:
df2 = df.copy()

# Here I normalize any known column variants, including split-header names
if 'OriginalImagePixelSpacing_x' in df2.columns or 'OriginalImagePixelSpacing[x' in df2.columns:
    df2 = unify_known_columns(df2)
    df2 = fix_split_bracket_columns(df2)

# Canonicalize pixel spacing columns from either normalized or split-header names
rename_map = {}
for col in df2.columns:
    if col in {'OriginalImagePixelSpacing[x', 'OriginalImagePixelSpacing_x'}:
        rename_map[col] = 'OriginalImagePixelSpacing_x'
    if col in {'y]', 'OriginalImagePixelSpacing_y'}:
        rename_map[col] = 'OriginalImagePixelSpacing_y'
if rename_map:
    df2 = df2.rename(columns=rename_map)

x_col = 'OriginalImagePixelSpacing_x' if 'OriginalImagePixelSpacing_x' in df2.columns else None
y_col = 'OriginalImagePixelSpacing_y' if 'OriginalImagePixelSpacing_y' in df2.columns else None

if x_col is None or y_col is None:
    print('Pixel spacing columns were not found in the dataframe.')
    print('Available matching columns:', [c for c in df2.columns if 'Pixel' in c or 'pixel' in c or 'Spacing' in c or 'spacing' in c])
else:
    df2 = df2.dropna(subset=[x_col, y_col])
    df2[x_col] = pd.to_numeric(df2[x_col], errors='coerce')
    df2[y_col] = pd.to_numeric(df2[y_col], errors='coerce')
    df2 = df2.dropna(subset=[x_col, y_col])

    # Plot of pixel level assessment
    plt.figure(figsize=(6,4))
    plt.scatter(df2[x_col], df2[y_col], alpha=0.5)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title('Pixel Spacing Distribution')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

In [ ]:
# To compare distributions across diseases and pixels
# First, I build a plotting dataframe with the canonical pixel-spacing columns
plot_df = df.copy()
plot_df['n_labels'] = plot_df['label_list'].apply(lambda lst: 0 if (len(lst) == 1 and lst[0] == 'No Finding') else len(lst))

# I add pixel spacing columns to the plotting dataframe if present
for col in ['OriginalImagePixelSpacing_x', 'OriginalImagePixelSpacing_y']:
    if col in df.columns:
        plot_df[col] = df[col]
    else:
        # fall back to the split-header names if they exist in the original dataframe
        if 'OriginalImagePixelSpacing[x' in df.columns:
            plot_df['OriginalImagePixelSpacing_x'] = pd.to_numeric(df['OriginalImagePixelSpacing[x'], errors='coerce')
        if 'y]' in df.columns:
            plot_df['OriginalImagePixelSpacing_y'] = pd.to_numeric(df['y]'], errors='coerce')

plt.figure(figsize=(8,4))
if 'OriginalImagePixelSpacing_x' in plot_df.columns:
    sns.boxplot(x='n_labels', y='OriginalImagePixelSpacing_x', data=plot_df)
    plt.title('OriginalImagePixelSpacing_x distribution by number of labels')
    plt.xlabel('Number of labels (0 = No Finding)')
    plt.ylabel('OriginalImagePixelSpacing_x')
    plt.show()
else:
    print('OriginalImagePixelSpacing_x is not available in the dataframe for plotting.')

plt.figure(figsize=(8,4))
if 'OriginalImagePixelSpacing_y' in plot_df.columns:
    sns.boxplot(x='n_labels', y='OriginalImagePixelSpacing_y', data=plot_df)
    plt.title('OriginalImagePixelSpacing_y distribution by number of labels')
    plt.xlabel('Number of labels (0 = No Finding)')
    plt.ylabel('OriginalImagePixelSpacing_y')
    plt.show()
else:
    print('OriginalImagePixelSpacing_y is not available in the dataframe for plotting.')

plt.figure(figsize=(8,4))
sns.boxplot(x='n_labels', y='Patient Age', data=plot_df)
plt.title('Patient Age distribution by number of labels')
plt.xlabel('Number of labels (0 = No Finding)')
plt.ylabel('Patient Age')
plt.show()

In [ ]:
## Visualize medical images using imshow

# Import required libraries
import pydicom
from glob import glob

# Create 'Pneumonia' column if it doesn't exist
if 'Pneumonia' not in all_xray_df.columns and 'Finding Labels' in all_xray_df.columns:
    all_xray_df['Pneumonia'] = all_xray_df['Finding Labels'].map(lambda x: 1.0 if 'Pneumonia' in x else 0.0)
    print('Created Pneumonia column from Finding Labels')

# Find DICOM files in current directory
dcm_files = sorted(glob('./*.dcm'))
print(f'Found {len(dcm_files)} DICOM files to visualize')

if len(dcm_files) > 0:
    # Determine grid size for displaying images
    n_images = len(dcm_files)
    n_cols = min(3, n_images)  # Max 3 columns
    n_rows = (n_images + n_cols - 1) // n_cols  # Calculate rows needed
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
    
    # Flatten axes for easier iteration if it's a single row/column
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    elif n_rows == 1 or n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    # Load and display each DICOM image
    for idx, dcm_file in enumerate(dcm_files):
        try:
            ds = pydicom.dcmread(dcm_file)
            img = ds.pixel_array.astype(np.float32)
            filename = os.path.basename(dcm_file)
            
            # Check if this file matches any Image Index in the dataset
            matching_rows = all_xray_df[
                all_xray_df['Image Index'].str.contains(filename.replace('.dcm', ''), na=False, case=False)
            ]
            
            # Determine label for title
            if len(matching_rows) > 0:
                is_pneumonia = matching_rows.iloc[0]['Pneumonia']
                label = 'Pneumonia' if is_pneumonia == 1.0 else 'Healthy'
                finding = matching_rows.iloc[0]['Finding Labels']
                title = f'{filename}\n{label}\n({finding[:30]}...)'
            else:
                title = f'{filename}\n(Test - Unlabeled)'
            
            # Display image using imshow
            axes[idx].imshow(img, cmap='gray')
            axes[idx].set_title(title, fontsize=10)
            axes[idx].axis('off')
            
            # Print basic image info
            print(f'{filename}: Shape={img.shape}, Min={img.min():.2f}, Max={img.max():.2f}, Mean={img.mean():.2f}')
            
        except Exception as e:
            axes[idx].text(0.5, 0.5, f'Error loading\n{os.path.basename(dcm_file)}', 
                          ha='center', va='center', fontsize=10)
            axes[idx].axis('off')
            print(f'Error loading {os.path.basename(dcm_file)}: {str(e)}')
    
    # Hide any unused subplots
    for idx in range(len(dcm_files), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f'\nDisplayed {len(dcm_files)} DICOM images')
    
    # Calculate intensity statistics across all images
    all_intensities = []
    for dcm_file in dcm_files:
        try:
            ds = pydicom.dcmread(dcm_file)
            img = ds.pixel_array.astype(np.float32)
            all_intensities.extend(img.flatten())
        except:
            pass
    
    if all_intensities:
        print(f'\nOverall intensity statistics across all images:')
        print(f'  Mean: {np.mean(all_intensities):.2f}')
        print(f'  Std: {np.std(all_intensities):.2f}')
        print(f'  Min: {np.min(all_intensities):.2f}')
        print(f'  Max: {np.max(all_intensities):.2f}')
        print(f'  Median: {np.median(all_intensities):.2f}')

else:
    print('No DICOM files found in current directory.')

## Findings and Implications for Model Training

### Class Balance

**Observation:** The dataset contains 1,430 pneumonia-positive studies out of 112,104 total studies, yielding a prevalence of **1.28%**. This means a baseline classifier that predicts "no pneumonia" for all inputs would achieve **98.72% accuracy**.

**Decision:** 
- Plain accuracy is rejected as the training metric and will be replaced by **F1-score, precision, recall, and AUC-ROC** for model evaluation.
- The training data must be rebalanced via undersampling the negative class or class weighting to prevent the model from learning a trivial "always negative" decision boundary.
- A patient-grouped stratified train/validation split will preserve class balance on both sides of the split.
- The final threshold will be selected by optimizing **F1-score** on the validation set rather than accuracy.

### Comorbidities

**Observation:** Pneumonia cases frequently co-occur with other thoracic findings:
- **Infiltration**: 514 cases (36.0% of pneumonia studies)
- **Edema**: 201 cases (14.1%)
- **Effusion**: 110 cases (7.7%)
- **Atelectasis**: 89 cases (6.2%)

Many pneumonia-positive studies have multiple concurrent findings, indicating complex clinical presentations.

**Decision:** 
- The negative class (non-pneumonia) must include a representative sample of diseased-but-not-pneumonia images (e.g., cases with Infiltration, Edema, Effusion, or Atelectasis but without Pneumonia label).
- Without this mix, the model risks learning "any abnormality = pneumonia" rather than learning pneumonia-specific features.
- During stratified splitting, the pneumonia label (not just case count) will be used to ensure both train and validation sets contain similar distributions of comorbid findings.

### Demographics and Views

**Observation:** After cleaning invalid ages (outliers > 120 or < 0):
- **Age**: Mean = 46.9 years; Interquartile range = 35–59 years; Range = 1–95 years
- **Gender**: Approximately 60% male, 40% female (consistent distribution across views)
- **View Position**: 
  - PA (Posteroanterior) view: 0.94% pneumonia prevalence
  - AP (Anteroposterior) view: 1.79% pneumonia prevalence
  - Other views (LL, RL, etc.): Rare

**Decision:**
- The **intended-use population** will be stated as frontal-view chest X-rays from adults, with emphasis on patients aged 20–80 years, since the pediatric tail (age < 10) is small and the very elderly (age > 85) show lower prevalence.
- The model is trained on both PA and AP views, so it should perform reasonably on both, though the inference pipeline will accept any frontal view.
- The validation dataset should maintain gender parity similar to the training set to avoid gender-specific model bias.

### Patient Overlap

**Observation:** Analysis of studies per patient reveals:
- **18 patients** have 100+ studies each
- **30 patients** have 75–99 studies
- **30,802 patients** in total have at least one study

This indicates significant **patient-level clustering**: multiple images from the same patient would artificially inflate train/validation overlap if not controlled.

**Decision:**
- The train/validation split **must be stratified at the patient level** (split by unique `Patient ID` first) to ensure no patient appears on both sides of the split.
- Class balance stratification (ensuring similar pneumonia prevalence in train and validation) will be applied *after* patient stratification.
- This patient-level split prevents information leakage and ensures the model generalizes to new patients, not just new images from known patients.